In [ ]:
https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/

In [ ]:

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor

# from .nn_wrappers import SklearnPyTorchNN, SklearnLassoNet
import sys
# sys.path not needed after pip install qtml
from NN_Regressor import NN_Regressor




import warnings
warnings.filterwarnings("ignore", category=UserWarning)



MODELS_REGRESSION = {
    "KNN": dict(model=KNeighborsRegressor(),
            params={"n_neighbors": range(3, 21)}, n_jobs=-1),

    "Ridge (L2)": dict(model=Ridge(),
              params={"alpha": [0.001, 0.01, 0.1, 1, 10, 100]}, n_jobs=-1),

    "Lasso (L1)": dict(model=Lasso(max_iter=1000000),
              params={"alpha": [0.001, 0.01, 0.1, 1, 10]}, n_jobs=-1,search="random"),

    "ElasticNet": dict(model=ElasticNet(max_iter=1000000),
                   params={"alpha": [0.001, 0.01, 0.1, 1, 10],
                    "l1_ratio": [0.2, 0.5, 0.8]}, n_jobs=-1),

    "RandomForest": dict(model=RandomForestRegressor(random_state=42),
                     params={"n_estimators": [100, 300],
                      "max_depth": [5, 10, None],
                      "min_samples_leaf": [1, 5]}, n_jobs=-1),

    "GBM": dict(model=GradientBoostingRegressor(random_state=42),
            params={"n_estimators": [100, 300],
             "max_depth": [3, 5],
             "learning_rate": [0.01, 0.1]}, n_jobs=-1),

    "XGBoost": dict(model=XGBRegressor(random_state=42),
                params={"n_estimators": [100, 300],
                 "max_depth": [3, 5, 7],
                 "learning_rate": [0.01, 0.1],
                 "subsample": [0.8, 1.0]}, n_jobs=-1),

    "LightGBM": dict(model=LGBMRegressor(verbose=-1, random_state=42),
                 params={"n_estimators": [100, 300],
                  "max_depth": [3, 5, -1],
                  "learning_rate": [0.01, 0.1],
                  "num_leaves": [31, 63]}, n_jobs=-1),

"CatBoost": dict(model=CatBoostRegressor(verbose=0, random_state=42),
                 params={"iterations": [300, 500],
                         "depth": [4, 6, 8],
                         "learning_rate": [0.01, 0.1]}, n_jobs=-1),

    # ── PyTorch NN ──
    #"PyTorch NN": dict(model=NN_Regressor(),
    #               params={"hidden_dims": [(32,), (64,), (64, 32), (128, 64)],
    #                "lr": [0.001], # "lr": [0.001, 0.01],
    #                "weight_decay": [0, 0.01],
    #                "l1_lambda": [0, 0.001],
    #                "dropout": [0.3, 0.4, 0.5, 0.55, 0.7],
    #                "n_epochs": [100]}, n_jobs=1, search="random"),



    
}












models = MODELS_REGRESSION

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import skew

# ── Kaggle CSVs laden ──
train = pd.read_csv("./data/train.csv")
test  = pd.read_csv("./data/test.csv")

# ── IDs sichern ──
train_id = train["Id"]
test_id  = test["Id"]
y = np.log(train["SalePrice"])

# ── Outlier NUR in Train entfernen ──
outlier_idx = train[train["GrLivArea"] > 4000].index
train = train.drop(outlier_idx)
y = y.drop(outlier_idx)

# ── Id + Target entfernen ──
train.drop(["Id", "SalePrice"], axis=1, inplace=True)
test.drop(["Id"], axis=1, inplace=True)

# ── Train + Test zusammen preprocessen ──
n_train = len(train)
df = pd.concat([train, test], axis=0).reset_index(drop=True)

# ── Missing Values ──
# Kategorisch: NaN hat Bedeutung ("kein Keller", "keine Garage" etc.)
fill_none = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
             "BsmtFinType2", "FireplaceQu", "GarageType", "GarageFinish",
             "GarageQual", "GarageCond", "PoolQC", "Fence", "MiscFeature"]
for col in fill_none:
    df[col] = df[col].fillna("Missing")

# Kategorisch: Rest mit Mode
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Numerisch: mit Median
df = df.fillna(df.median(numeric_only=True))

# ── Ordinale Kodierung (Qualitäts-Features) ──
qual_map = {"Missing": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
qual_cols = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
             "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual",
             "GarageCond", "PoolQC"]
for col in qual_cols:
    if col in df.columns:
        df[col] = df[col].map(qual_map).fillna(0).astype(int)

df["BsmtExposure"] = df["BsmtExposure"].map(
    {"Missing": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}).fillna(0).astype(int)
df["BsmtFinType1"] = df["BsmtFinType1"].map(
    {"Missing": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}).fillna(0).astype(int)
df["BsmtFinType2"] = df["BsmtFinType2"].map(
    {"Missing": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}).fillna(0).astype(int)
df["GarageFinish"] = df["GarageFinish"].map(
    {"Missing": 0, "Unf": 1, "RFn": 2, "Fin": 3}).fillna(0).astype(int)
df["Fence"] = df["Fence"].map(
    {"Missing": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4}).fillna(0).astype(int)
df["LotShape"] = df["LotShape"].map(
    {"IR3": 1, "IR2": 2, "IR1": 3, "Reg": 4}).fillna(3).astype(int)
df["LandSlope"] = df["LandSlope"].map(
    {"Sev": 1, "Mod": 2, "Gtl": 3}).fillna(3).astype(int)
df["CentralAir"] = df["CentralAir"].map({"N": 0, "Y": 1}).fillna(1).astype(int)
df["PavedDrive"] = df["PavedDrive"].map(
    {"N": 0, "P": 1, "Y": 2}).fillna(2).astype(int)

# ── Feature Engineering ──
df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
df["GarageAge"] = df["YrSold"] - df["GarageYrBlt"]
df["TotalBath"] = (df["FullBath"] + 0.5 * df["HalfBath"] +
                   df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"])
df["TotalPorch"] = (df["OpenPorchSF"] + df["EnclosedPorch"] +
                    df["3SsnPorch"] + df["ScreenPorch"])
df["BsmtFinSF"] = df["BsmtFinSF1"] + df["BsmtFinSF2"]
# Vor dem One-Hot Encoding hinzufügen:
df["OverallQual_x_GrLivArea"] = df["OverallQual"] * df["GrLivArea"]
df["OverallQual_x_TotalSF"] = df["OverallQual"] * df["TotalSF"]
df["OverallQual_sq"] = df["OverallQual"] ** 2
df["TotalSF_x_OverallCond"] = df["TotalSF"] * df["OverallCond"]

# ── Log-Transformation auf schiefe Features (nur nicht-negative) ──
numeric_cols = df.select_dtypes(include=[np.number]).columns
skewed_features = numeric_cols[df[numeric_cols].apply(lambda x: skew(x.dropna())).abs() > 0.5]
for col in skewed_features:
    if df[col].min() >= 0:
        df[col] = np.log1p(df[col])

# ── Dummy Encoding (nur für verbleibende kategorische) ──
df = pd.get_dummies(df, drop_first=True).astype(float)


# ── Sicherheit: Inf und NaN bereinigen ──
df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(df.median(numeric_only=True))

# ── Split zurück in Train und Test ──
X = df.iloc[:n_train]
X_new = df.iloc[n_train:]

# ── Skalieren ──
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_new = scaler.transform(X_new)

y = y.values  # Reset Index → numpy array

print(X.shape, y.shape, X_new.shape)
# Erwartet: (1458, ~220) (1458,) (1459, ~220)


(1456, 216) (1456,) (1459, 216)


In [32]:

import sys
# sys.path not needed after pip install qtml
from qtml.Cross_Sectional import run

# X Should be Scaled, y however must not be scaled!
results = run(X, y, models)

----- Model: KNN ------------------------------------------------------------------------------- 
--------------------------------------------------------------------------------------------------- 
Best: {'n_neighbors': 8}
KNN: Best={'n_neighbors': 8}, RMSE=0.1796
        RMSE: 0.179625
    Rsquared: 0.800991
         MAE: 0.120040
----- Model: Ridge (L2) ------------------------------------------------------------------------------- 
--------------------------------------------------------------------------------------------------- 
Best: {'alpha': 100}
Ridge (L2): Best={'alpha': 100}, RMSE=0.1232
        RMSE: 0.123168
    Rsquared: 0.901258
         MAE: 0.085718
----- Model: Lasso (L1) ------------------------------------------------------------------------------- 
--------------------------------------------------------------------------------------------------- 
Best: {'alpha': 0.001}
Lasso (L1): Best={'alpha': 0.001}, RMSE=0.1210
        RMSE: 0.120992
    Rsquared: 0.904717
  

In [ ]:
# Submission (Kaggle erwartet SalePrice)
best_model = results["CatBoost"]  # oder bestes Modell
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": np.exp(best_model.predict(X_new)) # best_model.predict(X_new)
})
submission.to_csv("./data/submission.csv", index=False)
print("submission.csv gespeichert!")

submission.csv gespeichert!


In [ ]:
# --- Ensemble: Blending --------------------------------------------------------------------

'''
pred_catboost = results["CatBoost"].predict(X_new)
pred_xgb      = results["XGBoost"].predict(X_new)
pred_gbm      = results["GBM"].predict(X_new)
pred_lgbm     = results["LightGBM"].predict(X_new)
pred_nn     = results["PyTorch NN"].predict(X_new)


# Blending — einfacher Durchschnitt
pred_blend = np.exp((pred_catboost + pred_xgb + pred_gbm + pred_lgbm + pred_nn) / 5)


# Submission
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": pred_blend  # Zurück von log → echte Preise
})
submission.to_csv("./data/submission.csv", index=False)
'''


# --- Ensemble: Voting ---
from sklearn.ensemble import VotingRegressor
from sklearn.model_selection import cross_val_score



voting = VotingRegressor(
    estimators=[
        ("CatBoost", results["CatBoost"]),
        ("XGBoost", results["XGBoost"]),
        ("GBM", results["GBM"]),
        ("LightGBM", results["LightGBM"]),
    ]
)

# Evaluieren
score = cross_val_score(voting, X, y, cv=5, scoring="neg_root_mean_squared_error")
print(f"Voting RMSLE: {-score.mean():.5f} ± {score.std():.5f}")

# Fitten + Submission
voting.fit(X, y)
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": np.exp(voting.predict(X_new))
})
submission.to_csv("./data/submission.csv", index=False)

Voting RMSLE: 0.11754 ± 0.00714


In [1]:
# --- Ensemble: Stacking --------------------------------------------------------------------

from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor


estimators = [
    ("CatBoost", CatBoostRegressor(
        depth=results["CatBoost"].get_params()["depth"],
        iterations=5000, learning_rate=0.02, verbose=0, random_state=42)),

    ("XGBoost", XGBRegressor(
        max_depth=results["XGBoost"].get_params()["max_depth"],
        subsample=results["XGBoost"].get_params()["subsample"],
        colsample_bytree=results["XGBoost"].get_params().get("colsample_bytree", 0.5),
        n_estimators=5000, learning_rate=0.02, random_state=42)),

    ("LightGBM", LGBMRegressor(
        max_depth=results["LightGBM"].get_params()["max_depth"],
        num_leaves=results["LightGBM"].get_params()["num_leaves"],
        n_estimators=5000, learning_rate=0.02, verbose=-1, random_state=42)),

    ("GBM", GradientBoostingRegressor(
        max_depth=results["GBM"].get_params()["max_depth"],
        n_estimators=3000, learning_rate=0.02, random_state=42)),

    ("ElasticNet", ElasticNet(
    alpha=results["ElasticNet"].get_params()["alpha"],
    l1_ratio=results["ElasticNet"].get_params()["l1_ratio"],
    max_iter=10000)),

    ("Lasso", Lasso(
    alpha=results["Lasso (L1)"].get_params()["alpha"],
    max_iter=10000)),

    ("Ridge", Ridge(
        alpha=results["Ridge (L2)"].get_params()["alpha"])),


]

stack = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5, n_jobs=-1
)


score = cross_val_score(stack, X, y, cv=5, scoring="neg_root_mean_squared_error")
print(f"Stacking RMSLE: {-score.mean():.5f} ± {score.std():.5f}")

stack.fit(X, y)
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": np.exp(stack.predict(X_new))
})
submission.to_csv("./data/submission.csv", index=False)

NameError: name 'results' is not defined